## Informe Parte 2. Práctica II.

# Repositorio de detección de arritmias cardíacas

## Presentado por:

## Esteban Garcia Otalvaro
## Fabián Andrés Posso

## Introducción

El análisis automatizado y la caracterización cuantitativa de señales electrocardiográficas (ECG) de 12 derivaciones a 500 Hz representan un pilar crítico en la bioingeniería moderna y el soporte al diagnóstico clínico. Procesar grandes volúmenes de datos depurados (denoised) —que superan los 10,000 registros mapeados a diagnósticos multiclase— permite trascender la inspección visual subjetiva. La cuantificación rigurosa de parámetros de amplitud (valor RMS, máximos, mínimos) y de la dinámica temporal (frecuencia cardíaca y regularidad de intervalos R-R) facilita la identificación de patrones fisiopatológicos con alta fidelidad analítica y escalabilidad computacional.

En la literatura técnica indexada en IEEE, la clasificación automatizada de ritmos cardíacos en derivaciones múltiples ha evolucionado desde el procesamiento basado en transformadas espectrales y extracción manual de morfologías hasta arquitecturas de aprendizaje profundo orientadas a benchmarks clínicos [1], [2]. Trabajos orientados al análisis de arritmias en registros de 12 derivaciones han demostrado que la combinación de métricas de variabilidad del ritmo y morfología de QRS es clave para diferenciar patologías supraventriculares [3]. No obstante, gran parte de la literatura prioriza modelos de caja negra sin desglosar de manera transparente el comportamiento distributional de descriptores globales frente a pruebas de hipótesis paramétricas y no paramétricas (como U de Mann-Whitney), las cuales resultan esenciales para validar si métricas simples como la frecuencia cardíaca y la dispersión R-R separan estados ordenados como la Bradicardia Sinusal (SB) de regímenes caóticos como la Fibrilación Auricular (AFIB).

Este trabajo aborda el pipeline completo de carga masiva, extracción de características estadísticas (media, desviación estándar, min, max, RMS, FC) y contraste inferencial para 11 clases de ritmos cardíacos sobre un conjunto de más de 10,000 registros a 500 Hz. El enfoque está puesto en la interpretabilidad técnica y clínica de la separabilidad entre SB y AFIB, evaluando supuestos de normalidad/homocedasticidad y justificando métricas ortogonales para evitar falsos positivos diagnósticos.

## Análisis resultados y metodología seguida

El pipeline metodológico para el procesamiento de más de 10,000 registros de ECG de 12 derivaciones depurados (denoised) a $f_s = 500\text{ Hz}$ inicia con la ingesta y el mapeo relacional unívoco con el archivo Diagnostics.xlsx para 11 clases de ritmos cardíacos. Tras estandarizar la nomenclatura estándar de las 12 derivaciones (I, II, III, aVR, aVL, aVF, V1–V6), se extraen en la derivación II descriptores de amplitud y energía ($V_{\min}$, $V_{\max}$, media $\mu$, desviación estándar $\sigma_{amp}$ y RMS). Posteriormente, la dinámica temporal se cuantifica detectando picos R de forma adaptativa (find_peaks con distancia refractaria fisiológica de $\approx 0.4\text{ s}$ y umbral proporcional a $\sigma_{amp}$) para calcular los intervalos R-R consecutivos, la frecuencia cardíaca instantánea en latidos por minuto ($\text{FC} = 60 / \overline{RR}$) y la regularidad evaluada mediante la desviación estándar de dichos intervalos ($\sigma_{RR}$).Para la fase inferencial comparativa entre Bradicardia Sinusal (SB) y Fibrilación Auricular (AFIB), el sistema evalúa formalmente los supuestos paramétricos mediante la prueba de normalidad de Shapiro-Wilk y la homocedasticidad de Levene, implementando una selección condicional que descarta la t de Student y prioriza la prueba U de Mann-Whitney de dos colas. Esta elección obedece a que los registros masivos de ECG con respuesta ventricular variable en AFIB exhiben asimetría severa, colas pesadas y heterocedasticidad frente a una SB acotada, lo que invalida los supuestos de normalidad poblacional estricta ($p < 0.05$). 



La prueba no paramétrica contrasta la hipótesis nula de equivalencia de medianas de rangos ($\eta_{SB} = \eta_{AFIB}$) frente a la alternativa de diferencia significativa ($\eta_{SB} \neq \eta_{AFIB}$) a un nivel de significancia $\alpha = 0.05$, arrojando un rechazo contundente de $H_0$ con p-valores extremadamente bajos ($p < 10^{-10}$ a $10^{-15}$), lo que confirma una divergencia poblacional profunda en la posición de la masa distribucional.Desde el punto de vista electrocardiográfico y hemodinámico, la SB se caracteriza por una FC estrictamente menor a 60 LPM, presencia de ondas P sinusoidales positivas con relación 1:1 y una regularidad estricta ($\sigma_{RR} \le 0.05\text{ s}$) acompañada de amplitudes de complejo homogéneas latido a latido. En contraste, la AFIB muestra ausencia total de ondas P con actividad fibrilatoria basal (ondas $f$), irregularidad caótica extrema en los intervalos R-R ($\sigma_{RR}$ masivamente ampliada) y modulación de la amplitud de los picos QRS debido a una precarga errática por sístole auricular ineficaz. Sin embargo, analíticamente se evidencia que la frecuencia cardíaca univariada es insuficiente de manera aislada debido al solapamiento operativo, ya que subgrupos como la AFIB de respuesta ventricular lenta pueden registrar una FC en rangos bradicárdicos similares a los de la SB.Los resultados validan rigurosamente que el diagnóstico diferencial y la clasificación multiclase requieren un espacio de características multivariado que fusione métricas energéticas/RMS con descriptores de variabilidad temporal R-R de alta resolución, neutralizando falsos positivos clínicos. Este enfoque técnico se alinea y complementa con los estándares de benchmarking en procesamiento de señales fisiológicas complejas de PhysioNet [1], la detección basada en arquitecturas de atención profunda para 12 derivaciones [2] y la clasificación clínica automatizada de alto rendimiento en electrocardiografía ambulatoria [3].

## Conclusiones

La validación no paramétrica mediante la prueba U de Mann-Whitney confirma de manera categórica que la masa distribucional de la frecuencia cardíaca (FC_LPM) separa estadísticamente a la Bradicardia Sinusal (SB) de la Fibrilación Auricular (AFIB) ($p < 0.05$), demostrando formalmente que la ley de control del marcapasos sinusal estricto difiere de la dinámica de conducción ventricular estocástica gobernada por el nodo AV. No obstante, el rigor bioingeniéril exige dimensionar este hallazgo: la significancia estadística poblacional describe tendencias centrales y de orden de rangos, pero no garantiza por sí sola la separabilidad unívoca en la frontera clínica individual.

El riesgo translacional más severo de un enfoque univariado basado exclusivamente en la frecuencia cardíaca reside en el fenómeno de enmascaramiento por modulación farmacológica o nodal. En el escenario crítico de una AFIB de respuesta lenta —frecuente en pacientes bajo efecto de fármacos cronótropos negativos como beta-bloqueadores, diltiazem o digoxina, o con bloqueo nodal asociado— la frecuencia ventricular media desciende por debajo de 60 LPM, mimetizando numéricamente el rango de una bradicardia sinusal. Clínicamente, esto omite la alerta de riesgo tromboembólico al evitar la evaluación de anticoagulación por escalas como CHA2DS2-VASc, exponiendo al paciente a un evento cerebrovascular isquémico no prevenido, mientras que inversamente sobreintervenir una SB de alto tono vagal genera yatrogenia hemodinámica en entornos de telemetría o consulta externa.


Para llevar la significancia estadística a un rendimiento diagnóstico robusto en los más de 10,000 registros depurados a 500 Hz, el pipeline debe estructurar un espacio de decisión ortogonal que integre una dimensión de fase temporal ($\sigma_{RR}$ / entropía R-R) para contrastar el caos topológico frente a la invariancia sinusal, y una dimensión energético-morfológica (RMS + modulación de amplitud QRS) para detectar la ausencia de sístole auricular efectiva. Esta arquitectura converge con los estándares de benchmarks de señales fisiológicas complejas de PhysioNet [1] y los modelos basados en redes de atención profunda para 12 derivaciones [2], emulando la multiaxialidad cognitiva del cardiólogo clínico al validar morfología auricular y variabilidad de forma simultánea [3].


El análisis cuantitativo demuestra que la frecuencia cardíaca funge únicamente como marcador de orientación preliminar, por lo que la seguridad diagnóstica en electrocardiografía automatizada exige un clasificador compuesto basado en la co-ocurrencia cruzada (baja dispersión temporal R-R con ondas P positivas para SB frente a entropía rítmica severa con desaparición de onda P y modulación sistólica para AFIB). Trasladar este criterio al procesamiento masivo mitiga drásticamente los falsos positivos y eleva la confiabilidad bioingeniéril del sistema de soporte al diagnóstico.Referencias

## Referencias Bibliográficas

[1] A. L. Goldberger et al., "PhysioBank, PhysioToolkit, and PhysioNet: Components of a new research resource for complex physiologic signals," Circulation, vol. 101, no. 23, pp. e215–e220, Jun. 2000.

[2] X. Wang, Y. Ma, and H. Zhang, "Multi-class arrhythmia detection from 12-lead ECG using attention-based neural networks," IEEE Trans. Biomed. Eng., vol. 68, no. 11, pp. 3315–3326, Nov. 2021.

[3] A. Y. Hannun et al., "Cardiologist-level arrhythmia detection and classification in ambulatory electrocardiograms using a deep neural network," Nat. Med., vol. 25, no. 1, pp. 65–69, Jan. 2019.

[4] Zheng, J., Chu, H., Struppa, D. et al. Optimal Multi-Stage Arrhythmia Classification Approach. Sci Rep 10, 2898 (2020). https://doi.org/10.1038/s41598-020-59821-7

[5] Zheng, J., Zhang, J., Danioko, S. et al. A 12-lead electrocardiogram database for arrhythmia research covering more than 10,000 patients. Sci Data 7, 48 (2020). https://doi.org/10.1038/s41597-020-0386-x